In [ ]:
import heapq
from collections import defaultdict, deque

# Graphs and Backtracking

---
# 1) Introduction

A graph question is solved in two steps: **model it** (what is a node, what is an edge, is
it weighted, directed?) and **pick the traversal** from the table below. Grids are graphs
whose nodes are cells and whose edges are the 4 neighbours.

| Need | Algorithm | Cost |
|---|---|---|
| reachability, components, flood fill | DFS or BFS | O(V + E) |
| fewest steps, unweighted | BFS (multi-source if several starts) | O(V + E) |
| ordering with prerequisites, cycle in directed graph | topological sort (Kahn) | O(V + E) |
| dynamic connectivity, merging groups, cycle in undirected graph | union–find | ~O(α(n)) per op |
| shortest path, non-negative weights | Dijkstra | O(E log V) |
| shortest path with ≤ k edges / negative weights | Bellman–Ford (k rounds) | O(k · E) |
| connect everything at min cost | Prim / Kruskal (MST) | O(E log V) |

**Backtracking** is DFS over a tree of *choices*: choose → recurse → un-choose. It
enumerates subsets, permutations, combinations and constraint-satisfaction solutions;
pruning invalid branches early is what makes it fast enough.

**How to use each question block.** Read the *Problem*, try for five minutes, then read the
*Signal*, then the *Idea*. Tiers: ★★★ Blind 75, ★★ NeetCode 150 / Grind 169 / Top Interview
150, ★ extended.

---
# 2) Grid Traversal

```python
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
def dfs(r, c):
    if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != target:
        return
    grid[r][c] = visited_mark          # mark before recursing
    for dr, dc in DIRS:
        dfs(r + dr, c + dc)
```

Python's default recursion limit is 1000; on large grids prefer an explicit stack or BFS.

In [ ]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

## 2.1 Connected regions (DFS)

### 2.1.1 [200. Number of Islands](https://leetcode.com/problems/number-of-islands/) — ★★★ · Medium

**Problem.** Count islands of `"1"` cells (4-directionally connected) in a grid.  
**Signal.** Count connected components → start a traversal from every unvisited land cell.  
**Idea.** Scan the grid; each time you meet unvisited land, increment the count and sink the whole island (mark visited) with DFS/BFS so it is not counted again.  
**Cost.** O(rows · cols) time and space

In [ ]:
def num_islands(grid: list[list[str]]) -> int:
    """Number of 4-connected land regions."""
    rows, cols = len(grid), len(grid[0])

    def sink(r: int, c: int) -> None:
        stack = [(r, c)]
        grid[r][c] = "0"
        while stack:
            r, c = stack.pop()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1":
                    grid[nr][nc] = "0"
                    stack.append((nr, nc))

    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1":
                count += 1
                sink(r, c)
    return count


g = [list("11000"), list("11000"), list("00100"), list("00011")]
assert num_islands(g) == 3
assert num_islands([list("11110"), list("11010"), list("11000"), list("00000")]) == 1
print("ok")

### 2.1.2 [695. Max Area of Island](https://leetcode.com/problems/max-area-of-island/) — ★★ · Medium

**Problem.** Area of the largest island (cells of value 1).  
**Signal.** Components with a size → DFS returning the count.  
**Idea.** DFS returns `1 + sum of neighbour areas`, marking cells as visited (set to 0).  
**Cost.** O(rows · cols)

In [ ]:
def max_area_of_island(grid: list[list[int]]) -> int:
    """Largest island area."""
    rows, cols = len(grid), len(grid[0])

    def area(r: int, c: int) -> int:
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != 1:
            return 0
        grid[r][c] = 0
        return 1 + sum(area(r + dr, c + dc) for dr, dc in DIRS)

    return max(area(r, c) for r in range(rows) for c in range(cols))


g = [
    [0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0],
    [0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
]
assert max_area_of_island(g) == 6
assert max_area_of_island([[0, 0]]) == 0
print("ok")

### 2.1.3 [733. Flood Fill](https://leetcode.com/problems/flood-fill/) — ★★ · Easy

**Problem.** Recolour the connected region of the start pixel's colour.  
**Signal.** Paint-bucket = DFS from one cell.  
**Idea.** Remember the original colour; if it already equals the new colour, return (otherwise the DFS would loop forever). Recolour and recurse into same-coloured neighbours.  
**Cost.** O(rows · cols)

In [ ]:
def flood_fill(image: list[list[int]], sr: int, sc: int, color: int) -> list[list[int]]:
    """Image after flood fill from (sr, sc)."""
    old = image[sr][sc]
    if old == color:
        return image
    rows, cols = len(image), len(image[0])

    def fill(r: int, c: int) -> None:
        if 0 <= r < rows and 0 <= c < cols and image[r][c] == old:
            image[r][c] = color
            for dr, dc in DIRS:
                fill(r + dr, c + dc)

    fill(sr, sc)
    return image


assert flood_fill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2) == [
    [2, 2, 2],
    [2, 2, 0],
    [2, 0, 1],
]
assert flood_fill([[0, 0, 0], [0, 0, 0]], 0, 0, 0) == [[0, 0, 0], [0, 0, 0]]
print("ok")

### 2.1.4 [463. Island Perimeter](https://leetcode.com/problems/island-perimeter/) — ★ · Easy

**Problem.** Perimeter of the single island in a grid.  
**Signal.** Count edges: each land cell adds 4, each shared edge removes 2.  
**Idea.** For every land cell add 4, subtract 2 for a land neighbour above and 2 for a land neighbour to the left (each shared edge is counted once this way).  
**Cost.** O(rows · cols)

In [ ]:
def island_perimeter(grid: list[list[int]]) -> int:
    """Perimeter of the island."""
    total = 0
    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell:
                total += 4
                if r and grid[r - 1][c]:
                    total -= 2
                if c and row[c - 1]:
                    total -= 2
    return total


assert island_perimeter([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]]) == 16
assert island_perimeter([[1]]) == 4
print("ok")

### 2.1.5 [130. Surrounded Regions](https://leetcode.com/problems/surrounded-regions/) — ★★ · Medium

**Problem.** Flip every `O` region not connected to the border into `X`.  
**Signal.** Invert the question: find what is *safe* (border-connected) and flip the rest.  
**Idea.** DFS from every border `O`, marking it `#`. Then turn remaining `O` into `X` and `#` back into `O`.  
**Cost.** O(rows · cols)

In [ ]:
def solve_surrounded(board: list[list[str]]) -> None:
    """Capture surrounded regions in place."""
    rows, cols = len(board), len(board[0])

    def mark(r: int, c: int) -> None:
        if 0 <= r < rows and 0 <= c < cols and board[r][c] == "O":
            board[r][c] = "#"
            for dr, dc in DIRS:
                mark(r + dr, c + dc)

    for r in range(rows):
        mark(r, 0)
        mark(r, cols - 1)
    for c in range(cols):
        mark(0, c)
        mark(rows - 1, c)
    for r in range(rows):
        for c in range(cols):
            board[r][c] = "O" if board[r][c] == "#" else "X"


b = [list("XXXX"), list("XOOX"), list("XXOX"), list("XOXX")]
solve_surrounded(b)
assert b == [list("XXXX"), list("XXXX"), list("XXXX"), list("XOXX")]
print(["".join(r) for r in b])

### 2.1.6 [417. Pacific Atlantic Water Flow](https://leetcode.com/problems/pacific-atlantic-water-flow/) — ★★★ · Medium

**Problem.** Cells from which water can flow (to equal or lower neighbours) into both the Pacific (top/left edges) and the Atlantic (bottom/right edges).  
**Signal.** Reverse the flow: climb *uphill* from each ocean's border and intersect.  
**Idea.** DFS from all Pacific border cells, moving to neighbours with height ≥ current; same for the Atlantic. Cells reached from both oceans are the answer. Two traversals instead of one per cell.  
**Cost.** O(rows · cols)

In [ ]:
def pacific_atlantic(heights: list[list[int]]) -> list[list[int]]:
    """Cells that drain to both oceans."""
    rows, cols = len(heights), len(heights[0])

    def climb(starts: list[tuple[int, int]]) -> set[tuple[int, int]]:
        seen = set(starts)
        stack = list(starts)
        while stack:
            r, c = stack.pop()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if (
                    0 <= nr < rows
                    and 0 <= nc < cols
                    and (nr, nc) not in seen
                    and heights[nr][nc] >= heights[r][c]
                ):
                    seen.add((nr, nc))
                    stack.append((nr, nc))
        return seen

    pacific = climb([(0, c) for c in range(cols)] + [(r, 0) for r in range(rows)])
    atlantic = climb(
        [(rows - 1, c) for c in range(cols)] + [(r, cols - 1) for r in range(rows)]
    )
    return sorted([r, c] for r, c in pacific & atlantic)


h = [
    [1, 2, 2, 3, 5],
    [3, 2, 3, 4, 4],
    [2, 4, 5, 3, 1],
    [6, 7, 1, 4, 5],
    [5, 1, 1, 2, 4],
]
assert pacific_atlantic(h) == [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
print(pacific_atlantic(h))

## 2.2 Shortest paths on grids (BFS)

BFS visits cells in order of distance, so the first time it reaches a cell is the shortest
way there. **Multi-source BFS** puts every start in the queue at distance 0 — it computes,
for every cell, the distance to the *nearest* source in one pass.

### 2.2.1 [994. Rotting Oranges](https://leetcode.com/problems/rotting-oranges/) — ★★ · Medium

**Problem.** Each minute rotten oranges rot their fresh neighbours. Minutes until none are fresh (−1 if impossible)?  
**Signal.** Simultaneous spreading from several sources → multi-source BFS by levels.  
**Idea.** Queue all rotten oranges, count fresh ones. Process level by level (one level = one minute), rotting fresh neighbours. If fresh oranges remain, return −1.  
**Cost.** O(rows · cols)

In [ ]:
def oranges_rotting(grid: list[list[int]]) -> int:
    """Minutes until every orange is rotten."""
    rows, cols = len(grid), len(grid[0])
    queue = deque((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == 2)
    fresh = sum(row.count(1) for row in grid)
    minutes = 0
    while queue and fresh:
        minutes += 1
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    queue.append((nr, nc))
    return -1 if fresh else minutes


assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
assert oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1
assert oranges_rotting([[0, 2]]) == 0
print("ok")

### 2.2.2 [286. Walls and Gates](https://leetcode.com/problems/walls-and-gates/) — ★★ · Medium · Premium

**Problem.** Fill each empty room (`INF`) with the distance to its nearest gate (`0`); walls are −1.  
**Signal.** Distance to nearest of many sources → multi-source BFS from all gates.  
**Idea.** Queue every gate. BFS assigns `dist + 1` to any neighbouring `INF` room; the first assignment is the shortest because BFS expands in distance order.  
**Cost.** O(rows · cols)

In [ ]:
INF = 2**31 - 1


def walls_and_gates(rooms: list[list[int]]) -> None:
    """Distance to nearest gate, in place."""
    rows, cols = len(rooms), len(rooms[0])
    queue = deque((r, c) for r in range(rows) for c in range(cols) if rooms[r][c] == 0)
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and rooms[nr][nc] == INF:
                rooms[nr][nc] = rooms[r][c] + 1
                queue.append((nr, nc))


rooms = [[INF, -1, 0, INF], [INF, INF, INF, -1], [INF, -1, INF, -1], [0, -1, INF, INF]]
walls_and_gates(rooms)
assert rooms == [[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]
print(rooms)

### 2.2.3 [542. 01 Matrix](https://leetcode.com/problems/01-matrix/) — ★★ · Medium

**Problem.** Distance of each cell to the nearest 0.  
**Signal.** Multi-source BFS from every 0.  
**Idea.** Initialise zeros with distance 0 and ones with "unvisited". BFS from all zeros; each unvisited neighbour gets `dist + 1`.  
**Cost.** O(rows · cols)

In [ ]:
def update_matrix(mat: list[list[int]]) -> list[list[int]]:
    """Distance from each cell to the nearest zero."""
    rows, cols = len(mat), len(mat[0])
    dist = [[0 if v == 0 else -1 for v in row] for row in mat]
    queue = deque((r, c) for r in range(rows) for c in range(cols) if mat[r][c] == 0)
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                queue.append((nr, nc))
    return dist


assert update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]) == [
    [0, 0, 0],
    [0, 1, 0],
    [1, 2, 1],
]
print(update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]))

### 2.2.4 [1091. Shortest Path in Binary Matrix](https://leetcode.com/problems/shortest-path-in-binary-matrix/) — ★ · Medium

**Problem.** Shortest 8-directional path of 0-cells from top-left to bottom-right (path length in cells), or −1.  
**Signal.** Unweighted shortest path → BFS with 8 neighbours.  
**Idea.** BFS from `(0, 0)` if it is open; mark cells visited when enqueued. The first time the target is popped gives the answer.  
**Cost.** O(n²)

In [ ]:
def shortest_path_binary_matrix(grid: list[list[int]]) -> int:
    """Length of the shortest clear path, or -1."""
    n = len(grid)
    if grid[0][0] or grid[-1][-1]:
        return -1
    queue = deque([(0, 0, 1)])
    grid[0][0] = 1
    while queue:
        r, c, d = queue.popleft()
        if (r, c) == (n - 1, n - 1):
            return d
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 0:
                    grid[nr][nc] = 1
                    queue.append((nr, nc, d + 1))
    return -1


assert shortest_path_binary_matrix([[0, 1], [1, 0]]) == 2
assert shortest_path_binary_matrix([[0, 0, 0], [1, 1, 0], [1, 1, 0]]) == 4
assert shortest_path_binary_matrix([[1, 0, 0], [1, 1, 0], [1, 1, 0]]) == -1
print("ok")

### 2.2.5 [1926. Nearest Exit from Entrance in Maze](https://leetcode.com/problems/nearest-exit-from-entrance-in-maze/) — ★ · Medium

**Problem.** Fewest steps from the entrance to any border empty cell (not the entrance itself).  
**Signal.** Unweighted shortest path to any of many targets → BFS, stop at the first target.  
**Idea.** BFS from the entrance; when dequeuing a border cell other than the entrance, return its distance.  
**Cost.** O(rows · cols)

In [ ]:
def nearest_exit(maze: list[list[str]], entrance: list[int]) -> int:
    """Steps to the nearest exit, or -1."""
    rows, cols = len(maze), len(maze[0])
    start = tuple(entrance)
    queue = deque([(start, 0)])
    maze[start[0]][start[1]] = "+"
    while queue:
        (r, c), d = queue.popleft()
        if (r, c) != start and (r in (0, rows - 1) or c in (0, cols - 1)):
            return d
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] == ".":
                maze[nr][nc] = "+"
                queue.append(((nr, nc), d + 1))
    return -1


assert nearest_exit([list("++.+"), list("...+"), list("+++.")], [1, 2]) == 1
assert nearest_exit([list("+++"), list("..."), list("+++")], [1, 0]) == 2
assert nearest_exit([list(".+")], [0, 0]) == -1
print("ok")

### 2.2.6 [934. Shortest Bridge](https://leetcode.com/problems/shortest-bridge/) — ★ · Medium

**Problem.** Exactly two islands; fewest 0-cells to flip to connect them.  
**Signal.** DFS to collect one island, then multi-source BFS outward from all its cells.  
**Idea.** Find the first island and mark it (value 2), queueing all its cells. BFS expands over water level by level; the level at which it touches the other island (value 1) is the answer.  
**Cost.** O(n²)

In [ ]:
def shortest_bridge(grid: list[list[int]]) -> int:
    """Zeros to flip to join the two islands."""
    n = len(grid)
    start = next((r, c) for r in range(n) for c in range(n) if grid[r][c])
    queue = deque()
    stack = [start]
    grid[start[0]][start[1]] = 2
    while stack:
        r, c = stack.pop()
        queue.append((r, c))
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 1:
                grid[nr][nc] = 2
                stack.append((nr, nc))
    steps = 0
    while queue:
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n:
                    if grid[nr][nc] == 1:
                        return steps
                    if grid[nr][nc] == 0:
                        grid[nr][nc] = 2
                        queue.append((nr, nc))
        steps += 1
    return -1


assert shortest_bridge([[0, 1], [1, 0]]) == 1
assert shortest_bridge([[0, 1, 0], [0, 0, 0], [0, 0, 1]]) == 2
g = [
    [1, 1, 1, 1, 1],
    [1, 0, 0, 0, 1],
    [1, 0, 1, 0, 1],
    [1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1],
]
assert shortest_bridge(g) == 1
print("ok")

---
# 3) Graph Traversal

Build an adjacency list first: `graph = defaultdict(list)`; for undirected edges add both
directions. Track `visited` to avoid infinite loops on cycles.

## 3.1 Adjacency-list traversal

### 3.1.1 [133. Clone Graph](https://leetcode.com/problems/clone-graph/) — ★★★ · Medium

**Problem.** Deep-copy a connected undirected graph given one node.  
**Signal.** Traverse the graph while mapping original → clone (the map doubles as `visited`).  
**Idea.** DFS: if a node is already cloned return the clone; otherwise create it, store it in the map *before* recursing (to handle cycles), then clone each neighbour.  
**Cost.** O(V + E)

In [ ]:
class Node:
    """Graph node with neighbours."""

    def __init__(self, val: int = 0, neighbors: list | None = None) -> None:
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []


def clone_graph(node: Node | None) -> Node | None:
    """Deep copy of the graph reachable from node."""
    clones: dict[Node, Node] = {}

    def dfs(n: Node) -> Node:
        if n in clones:
            return clones[n]
        copy = Node(n.val)
        clones[n] = copy
        copy.neighbors = [dfs(nb) for nb in n.neighbors]
        return copy

    return dfs(node) if node else None


nodes = {i: Node(i) for i in range(1, 5)}
for a, nbs in {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [1, 3]}.items():
    nodes[a].neighbors = [nodes[b] for b in nbs]
c = clone_graph(nodes[1])
assert c is not nodes[1] and [n.val for n in c.neighbors] == [2, 4]
assert [n.val for n in c.neighbors[0].neighbors] == [1, 3]
print("ok")

### 3.1.2 [547. Number of Provinces](https://leetcode.com/problems/number-of-provinces/) — ★ · Medium

**Problem.** Adjacency matrix of cities; count connected groups.  
**Signal.** Connected components on a matrix-encoded graph.  
**Idea.** DFS from each unvisited city over row `i` of the matrix; each start is a new province.  
**Cost.** O(n²)

In [ ]:
def find_circle_num(is_connected: list[list[int]]) -> int:
    """Number of connected components."""
    n = len(is_connected)
    seen = [False] * n

    def dfs(i: int) -> None:
        seen[i] = True
        for j in range(n):
            if is_connected[i][j] and not seen[j]:
                dfs(j)

    count = 0
    for i in range(n):
        if not seen[i]:
            count += 1
            dfs(i)
    return count


assert find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]) == 2
assert find_circle_num([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 3
print("ok")

### 3.1.3 [841. Keys and Rooms](https://leetcode.com/problems/keys-and-rooms/) — ★ · Medium

**Problem.** Room 0 is open; each room has keys to other rooms. Can you visit all rooms?  
**Signal.** Reachability from node 0.  
**Idea.** DFS from room 0 using keys as edges; compare the visited count with the number of rooms.  
**Cost.** O(V + E)

In [ ]:
def can_visit_all_rooms(rooms: list[list[int]]) -> bool:
    """True if every room is reachable from room 0."""
    seen, stack = {0}, [0]
    while stack:
        for key in rooms[stack.pop()]:
            if key not in seen:
                seen.add(key)
                stack.append(key)
    return len(seen) == len(rooms)


assert can_visit_all_rooms([[1], [2], [3], []])
assert not can_visit_all_rooms([[1, 3], [3, 0, 1], [2], [0]])
print("ok")

### 3.1.4 [323. Number of Connected Components in an Undirected Graph](https://leetcode.com/problems/number-of-connected-components-in-an-undirected-graph/) — ★★★ · Medium · Premium

**Problem.** `n` nodes and an edge list; count connected components.  
**Signal.** Components → DFS from each unvisited node (or union–find: `n − successful unions`).  
**Idea.** Build the adjacency list, DFS from every unvisited node, count the starts.  
**Cost.** O(V + E)

In [ ]:
def count_components(n: int, edges: list[list[int]]) -> int:
    """Number of connected components."""
    graph = defaultdict(list)
    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)
    seen: set[int] = set()
    count = 0
    for start in range(n):
        if start in seen:
            continue
        count += 1
        stack = [start]
        seen.add(start)
        while stack:
            for nb in graph[stack.pop()]:
                if nb not in seen:
                    seen.add(nb)
                    stack.append(nb)
    return count


assert count_components(5, [[0, 1], [1, 2], [3, 4]]) == 2
assert count_components(5, [[0, 1], [1, 2], [2, 3], [3, 4]]) == 1
print("ok")

### 3.1.5 [261. Graph Valid Tree](https://leetcode.com/problems/graph-valid-tree/) — ★★★ · Medium · Premium

**Problem.** Do `n` nodes and these undirected edges form a tree?  
**Signal.** A tree has exactly `n − 1` edges and is connected (equivalently: no cycle).  
**Idea.** If `len(edges) != n − 1`, it cannot be a tree. Otherwise it is a tree iff it is connected, i.e. one component.  
**Cost.** O(V + E)

In [ ]:
def valid_tree(n: int, edges: list[list[int]]) -> bool:
    """True if the graph is a tree."""
    return len(edges) == n - 1 and count_components(n, edges) == 1


assert valid_tree(5, [[0, 1], [0, 2], [0, 3], [1, 4]])
assert not valid_tree(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]])
print("ok")

### 3.1.6 [785. Is Graph Bipartite?](https://leetcode.com/problems/is-graph-bipartite/) — ★ · Medium

**Problem.** Can nodes be 2-coloured so every edge joins different colours?  
**Signal.** 2-colouring via BFS/DFS; conflict ⇔ odd cycle.  
**Idea.** Colour an uncoloured node 0, then give each neighbour the opposite colour. A neighbour already having the same colour proves the graph is not bipartite. Repeat per component.  
**Cost.** O(V + E)

In [ ]:
def is_bipartite(graph: list[list[int]]) -> bool:
    """True if the graph is 2-colourable."""
    color: dict[int, int] = {}
    for start in range(len(graph)):
        if start in color:
            continue
        color[start] = 0
        stack = [start]
        while stack:
            node = stack.pop()
            for nb in graph[node]:
                if nb not in color:
                    color[nb] = 1 - color[node]
                    stack.append(nb)
                elif color[nb] == color[node]:
                    return False
    return True


assert not is_bipartite([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]])
assert is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]])
print("ok")

### 3.1.7 [1466. Reorder Routes to Make All Paths Lead to the City Zero](https://leetcode.com/problems/reorder-routes-to-make-all-paths-lead-to-the-city-zero/) — ★ · Medium

**Problem.** Tree of directed roads; minimum number of roads to reverse so every city can reach 0.  
**Signal.** Traverse the undirected tree from 0, counting edges that point away from 0.  
**Idea.** Store each edge in both directions with a flag `1` for the original direction. DFS from 0: every original edge we traverse outward points away from 0 and must be reversed.  
**Cost.** O(n)

In [ ]:
def min_reorder(n: int, connections: list[list[int]]) -> int:
    """Number of edges to flip so all reach city 0."""
    graph = defaultdict(list)
    for a, b in connections:
        graph[a].append((b, 1))
        graph[b].append((a, 0))
    seen, stack, flips = {0}, [0], 0
    while stack:
        node = stack.pop()
        for nb, is_original in graph[node]:
            if nb not in seen:
                seen.add(nb)
                flips += is_original
                stack.append(nb)
    return flips


assert min_reorder(6, [[0, 1], [1, 3], [2, 3], [4, 0], [4, 5]]) == 3
assert min_reorder(5, [[1, 0], [1, 2], [3, 2], [3, 4]]) == 2
print("ok")

### 3.1.8 [399. Evaluate Division](https://leetcode.com/problems/evaluate-division/) — ★★ · Medium

**Problem.** Given `a / b = k` equations, answer queries `x / y` (or −1).  
**Signal.** Weighted graph: edge `a → b` with weight `k`, `b → a` with `1/k`; a query multiplies weights along a path.  
**Idea.** Build the graph. For each query, DFS from `x` multiplying weights until `y` is reached. Unknown variables return −1.  
**Cost.** O(Q · (V + E))

In [ ]:
def calc_equation(
    equations: list[list[str]], values: list[float], queries: list[list[str]]
) -> list[float]:
    """Answers to division queries."""
    graph: dict[str, list[tuple[str, float]]] = defaultdict(list)
    for (a, b), k in zip(equations, values):
        graph[a].append((b, k))
        graph[b].append((a, 1 / k))

    def dfs(x: str, y: str, seen: set[str]) -> float:
        if x == y:
            return 1.0
        seen.add(x)
        for nb, w in graph[x]:
            if nb not in seen:
                found = dfs(nb, y, seen)
                if found > 0:
                    return w * found
        return -1.0

    return [dfs(x, y, set()) if x in graph and y in graph else -1.0 for x, y in queries]


res = calc_equation(
    [["a", "b"], ["b", "c"]],
    [2.0, 3.0],
    [["a", "c"], ["b", "a"], ["a", "e"], ["a", "a"], ["x", "x"]],
)
assert res == [6.0, 0.5, -1.0, 1.0, -1.0]
print(res)

## 3.2 BFS over state spaces

When nodes are *states* (words, lock combinations, board squares), generate neighbours on
the fly instead of building the graph.

### 3.2.1 [127. Word Ladder](https://leetcode.com/problems/word-ladder/) — ★★ · Hard

**Problem.** Fewest words in a transformation sequence from `begin` to `end`, changing one letter at a time, each intermediate word in the list.  
**Signal.** Shortest path in an implicit graph → BFS; neighbours via wildcard patterns.  
**Idea.** Map each pattern like `h*t` to the words matching it. BFS from `begin`; neighbours of a word are all words sharing one of its patterns. Clear a pattern's list after use so each bucket is scanned once.  
**Cost.** O(N · L²) time

In [ ]:
def ladder_length(begin_word: str, end_word: str, word_list: list[str]) -> int:
    """Length of the shortest transformation sequence, or 0."""
    if end_word not in word_list:
        return 0
    buckets: dict[str, list[str]] = defaultdict(list)
    for w in word_list:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1 :]].append(w)
    queue, seen = deque([(begin_word, 1)]), {begin_word}
    while queue:
        word, steps = queue.popleft()
        if word == end_word:
            return steps
        for i in range(len(word)):
            pattern = word[:i] + "*" + word[i + 1 :]
            for nb in buckets[pattern]:
                if nb not in seen:
                    seen.add(nb)
                    queue.append((nb, steps + 1))
            buckets[pattern] = []
    return 0


words = ["hot", "dot", "dog", "lot", "log", "cog"]
assert ladder_length("hit", "cog", words) == 5
assert ladder_length("hit", "cog", words[:-1]) == 0
print(ladder_length("hit", "cog", words))

### 3.2.2 [433. Minimum Genetic Mutation](https://leetcode.com/problems/minimum-genetic-mutation/) — ★★ · Medium

**Problem.** Fewest single-character mutations (letters A/C/G/T) from `start` to `end`, each step in `bank`.  
**Signal.** Word Ladder with a 4-letter alphabet.  
**Idea.** BFS; neighbours are bank strings differing by one character (try all 8 × 3 mutations).  
**Cost.** O(8 · 4 · |bank|)

In [ ]:
def min_mutation(start: str, end: str, bank: list[str]) -> int:
    """Fewest mutations, or -1."""
    valid = set(bank)
    queue, seen = deque([(start, 0)]), {start}
    while queue:
        gene, steps = queue.popleft()
        if gene == end:
            return steps
        for i in range(len(gene)):
            for ch in "ACGT":
                nb = gene[:i] + ch + gene[i + 1 :]
                if nb in valid and nb not in seen:
                    seen.add(nb)
                    queue.append((nb, steps + 1))
    return -1


assert min_mutation("AACCGGTT", "AACCGGTA", ["AACCGGTA"]) == 1
assert min_mutation("AACCGGTT", "AAACGGTA", ["AACCGGTA", "AACCGCTA", "AAACGGTA"]) == 2
print("ok")

### 3.2.3 [752. Open the Lock](https://leetcode.com/problems/open-the-lock/) — ★ · Medium

**Problem.** 4-wheel lock from `"0000"`; each move turns one wheel by ±1; avoid `deadends`. Fewest moves to `target`?  
**Signal.** BFS over 10⁴ states.  
**Idea.** Each state has 8 neighbours. BFS from `"0000"` treating deadends as visited.  
**Cost.** O(10⁴ · 8)

In [ ]:
def open_lock(deadends: list[str], target: str) -> int:
    """Fewest turns to reach target, or -1."""
    seen = set(deadends)
    if "0000" in seen:
        return -1
    seen.add("0000")
    queue = deque([("0000", 0)])
    while queue:
        state, steps = queue.popleft()
        if state == target:
            return steps
        for i in range(4):
            d = int(state[i])
            for nd in ((d + 1) % 10, (d - 1) % 10):
                nb = state[:i] + str(nd) + state[i + 1 :]
                if nb not in seen:
                    seen.add(nb)
                    queue.append((nb, steps + 1))
    return -1


assert open_lock(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
assert open_lock(["8888"], "0009") == 1
assert (
    open_lock(["8887", "8889", "8878", "8898", "8788", "8988", "7888", "9888"], "8888")
    == -1
)
print("ok")

### 3.2.4 [909. Snakes and Ladders](https://leetcode.com/problems/snakes-and-ladders/) — ★★ · Medium

**Problem.** Boustrophedon-numbered board; roll 1–6 each move, snakes/ladders teleport. Fewest moves to the last square?  
**Signal.** BFS over squares; the tricky part is converting square number → (row, col).  
**Idea.** Square `s` (1-based) is at row `n − 1 − (s − 1) // n` from the top; columns run left to right on even rows from the bottom and right to left on odd ones. BFS from square 1; for each roll, follow a snake/ladder if present.  
**Cost.** O(n²)

In [ ]:
def snakes_and_ladders(board: list[list[int]]) -> int:
    """Fewest dice rolls to reach the final square."""
    n = len(board)

    def cell(s: int) -> int:
        q, r = divmod(s - 1, n)
        row = n - 1 - q
        col = r if q % 2 == 0 else n - 1 - r
        return board[row][col]

    queue, seen = deque([(1, 0)]), {1}
    while queue:
        s, moves = queue.popleft()
        if s == n * n:
            return moves
        for nxt in range(s + 1, min(s + 6, n * n) + 1):
            dest = cell(nxt) if cell(nxt) != -1 else nxt
            if dest not in seen:
                seen.add(dest)
                queue.append((dest, moves + 1))
    return -1


b = [
    [-1, -1, -1, -1, -1, -1],
    [-1, -1, -1, -1, -1, -1],
    [-1, -1, -1, -1, -1, -1],
    [-1, 35, -1, -1, 13, -1],
    [-1, -1, -1, -1, -1, -1],
    [-1, 15, -1, -1, -1, -1],
]
assert snakes_and_ladders(b) == 4
assert snakes_and_ladders([[-1, -1], [-1, 3]]) == 1
print("ok")

### 3.2.5 [815. Bus Routes](https://leetcode.com/problems/bus-routes/) — ★ · Hard

**Problem.** `routes[i]` is a bus loop over stops. Fewest buses to get from `source` to `target`?  
**Signal.** BFS where nodes are *buses*, not stops (one step = one bus ride).  
**Idea.** Map stop → buses. Start with all buses through `source`. Pop a bus, ride it to each of its stops; if a stop is the target, done; otherwise enqueue unused buses at that stop. Mark buses (and stops) visited to avoid rescanning.  
**Cost.** O(total stops)

In [ ]:
def num_buses_to_destination(routes: list[list[int]], source: int, target: int) -> int:
    """Fewest buses from source to target, or -1."""
    if source == target:
        return 0
    stop_to_buses: dict[int, list[int]] = defaultdict(list)
    for bus, route in enumerate(routes):
        for stop in route:
            stop_to_buses[stop].append(bus)
    queue = deque((bus, 1) for bus in stop_to_buses[source])
    used_bus = set(stop_to_buses[source])
    seen_stop = {source}
    while queue:
        bus, rides = queue.popleft()
        for stop in routes[bus]:
            if stop == target:
                return rides
            if stop in seen_stop:
                continue
            seen_stop.add(stop)
            for nb in stop_to_buses[stop]:
                if nb not in used_bus:
                    used_bus.add(nb)
                    queue.append((nb, rides + 1))
    return -1


assert num_buses_to_destination([[1, 2, 7], [3, 6, 7]], 1, 6) == 2
routes = [[7, 12], [4, 5, 15], [6], [15, 19], [9, 12, 13]]
assert num_buses_to_destination(routes, 15, 12) == -1
print("ok")

---
# 4) Topological Sort

For a directed graph, **Kahn's algorithm** repeatedly removes nodes with in-degree 0. The
removal order is a valid ordering; if some nodes are never removed, they sit on a cycle.

```python
indeg = [0] * n; graph = defaultdict(list)
for a, b in edges: graph[a].append(b); indeg[b] += 1
queue = deque(i for i in range(n) if indeg[i] == 0)
order = []
while queue:
    u = queue.popleft(); order.append(u)
    for v in graph[u]:
        indeg[v] -= 1
        if indeg[v] == 0: queue.append(v)
# len(order) < n  ⇒ cycle
```

## 4.1 Kahn's algorithm and DFS ordering

### 4.1.1 [207. Course Schedule](https://leetcode.com/problems/course-schedule/) — ★★★ · Medium

**Problem.** `prerequisites[i] = [a, b]` means take `b` before `a`. Can all courses be finished?  
**Signal.** Prerequisites → directed graph; feasible ⇔ no cycle ⇔ topological order covers all.  
**Idea.** Run Kahn's algorithm; return whether every course was removed.  
**Cost.** O(V + E)

In [ ]:
def topo_order(n: int, edges: list[tuple[int, int]]) -> list[int]:
    """Kahn's order for edges u -> v (shorter than n if cyclic)."""
    graph = defaultdict(list)
    indeg = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indeg[v] += 1
    queue = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    return order


def can_finish(num_courses: int, prerequisites: list[list[int]]) -> bool:
    """True if the prerequisite graph is acyclic."""
    edges = [(b, a) for a, b in prerequisites]
    return len(topo_order(num_courses, edges)) == num_courses


assert can_finish(2, [[1, 0]])
assert not can_finish(2, [[1, 0], [0, 1]])
print("ok")

### 4.1.2 [210. Course Schedule II](https://leetcode.com/problems/course-schedule-ii/) — ★★ · Medium

**Problem.** Return any valid order to take all courses, or `[]` if impossible.  
**Signal.** Same as Course Schedule, return the order itself.  
**Idea.** Kahn's order; if it is shorter than `n`, a cycle exists.  
**Cost.** O(V + E)

In [ ]:
def find_order(num_courses: int, prerequisites: list[list[int]]) -> list[int]:
    """A valid course order, or empty."""
    order = topo_order(num_courses, [(b, a) for a, b in prerequisites])
    return order if len(order) == num_courses else []


assert find_order(2, [[1, 0]]) == [0, 1]
order = find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]])
assert order[0] == 0 and order[-1] == 3
assert find_order(2, [[0, 1], [1, 0]]) == []
print(order)

### 4.1.3 [269. Alien Dictionary](https://leetcode.com/problems/alien-dictionary/) — ★★★ · Hard · Premium

**Problem.** Words sorted in an unknown alphabet; return a valid letter order (or `""` if invalid).  
**Signal.** Adjacent words give ordering constraints → topological sort on letters.  
**Idea.** For each adjacent pair, the first differing letter gives an edge `a → b`. If the first word is longer and the second is its prefix, the input is invalid. Topologically sort all letters; a cycle means no valid order.  
**Cost.** O(total characters)

In [ ]:
def alien_order(words: list[str]) -> str:
    """Letter order consistent with the sorted words."""
    letters = sorted({ch for w in words for ch in w})
    index = {ch: i for i, ch in enumerate(letters)}
    edges = set()
    for w1, w2 in zip(words, words[1:]):
        for a, b in zip(w1, w2):
            if a != b:
                edges.add((index[a], index[b]))
                break
        else:
            if len(w1) > len(w2):
                return ""
    order = topo_order(len(letters), sorted(edges))
    return "".join(letters[i] for i in order) if len(order) == len(letters) else ""


assert alien_order(["wrt", "wrf", "er", "ett", "rftt"]) == "wertf"
assert alien_order(["z", "x"]) == "zx"
assert alien_order(["z", "x", "z"]) == ""
assert alien_order(["abc", "ab"]) == ""
print(alien_order(["wrt", "wrf", "er", "ett", "rftt"]))

### 4.1.4 [310. Minimum Height Trees](https://leetcode.com/problems/minimum-height-trees/) — ★★ · Medium

**Problem.** Roots of all minimum-height trees of an undirected tree.  
**Signal.** Peel leaves layer by layer (topological sort on an undirected tree); the last 1–2 nodes are the centres.  
**Idea.** Repeatedly remove all current leaves (degree 1), updating neighbours' degrees. When at most two nodes remain, they are the centres of the longest path, hence the MHT roots.  
**Cost.** O(n)

In [ ]:
def find_min_height_trees(n: int, edges: list[list[int]]) -> list[int]:
    """Centres of the tree."""
    if n == 1:
        return [0]
    graph = defaultdict(set)
    for a, b in edges:
        graph[a].add(b)
        graph[b].add(a)
    leaves = [i for i in range(n) if len(graph[i]) == 1]
    remaining = n
    while remaining > 2:
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            nb = graph[leaf].pop()
            graph[nb].discard(leaf)
            if len(graph[nb]) == 1:
                nxt.append(nb)
        leaves = nxt
    return sorted(leaves)


assert find_min_height_trees(4, [[1, 0], [1, 2], [1, 3]]) == [1]
assert find_min_height_trees(6, [[3, 0], [3, 1], [3, 2], [3, 4], [5, 4]]) == [3, 4]
print("ok")

### 4.1.5 [802. Find Eventual Safe States](https://leetcode.com/problems/find-eventual-safe-states/) — ★ · Medium

**Problem.** A node is safe if every path from it ends at a terminal node. Return all safe nodes.  
**Signal.** Reverse the edges and run Kahn from terminal nodes (out-degree 0).  
**Idea.** In the reversed graph, terminal nodes have in-degree 0. Peeling them repeatedly marks every node whose all outgoing edges lead to safe nodes. Nodes on or leading into cycles never reach in-degree 0.  
**Cost.** O(V + E)

In [ ]:
def eventual_safe_nodes(graph: list[list[int]]) -> list[int]:
    """Nodes from which every path terminates."""
    reversed_edges = [(v, u) for u, outs in enumerate(graph) for v in outs]
    return sorted(topo_order(len(graph), reversed_edges))


assert eventual_safe_nodes([[1, 2], [2, 3], [5], [0], [5], [], []]) == [2, 4, 5, 6]
assert eventual_safe_nodes([[1, 2, 3, 4], [1, 2], [3, 4], [0, 4], []]) == [4]
print("ok")

---
# 5) Union-Find

Union–find (disjoint set union) keeps a forest where each tree is a group. With **path
compression** and **union by size**, each operation is effectively O(1). Use it when
edges arrive one at a time and you keep asking "are these two already connected?".

In [ ]:
class DSU:
    """Disjoint set union with path compression and union by size."""

    def __init__(self, n: int) -> None:
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> bool:
        """Merge groups; False if already joined."""
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


d = DSU(4)
print(d.union(0, 1), d.union(1, 0), d.find(1) == d.find(0))

## 5.1 Dynamic connectivity

### 5.1.1 [684. Redundant Connection](https://leetcode.com/problems/redundant-connection/) — ★★ · Medium

**Problem.** A tree plus one extra edge; return the edge that can be removed (the last such in input).  
**Signal.** The first edge joining two already-connected nodes closes the cycle → union–find.  
**Idea.** Union edges in order; the edge whose `union` returns `False` is redundant.  
**Cost.** O(n α(n))

In [ ]:
def find_redundant_connection(edges: list[list[int]]) -> list[int]:
    """Edge that creates the cycle."""
    dsu = DSU(len(edges) + 1)
    for a, b in edges:
        if not dsu.union(a, b):
            return [a, b]
    return []


assert find_redundant_connection([[1, 2], [1, 3], [2, 3]]) == [2, 3]
assert find_redundant_connection([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]) == [1, 4]
print("ok")

### 5.1.2 [721. Accounts Merge](https://leetcode.com/problems/accounts-merge/) — ★★ · Medium

**Problem.** Merge accounts that share any email; output name + sorted emails.  
**Signal.** "Share any element → same group" → union–find on account indices.  
**Idea.** Map each email to the first account that had it; union later accounts with that one. Then group emails by the root account.  
**Cost.** O(N α(N) + N log N) for N emails

In [ ]:
def accounts_merge(accounts: list[list[str]]) -> list[list[str]]:
    """Merged accounts with sorted emails."""
    dsu = DSU(len(accounts))
    owner: dict[str, int] = {}
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            if e in owner:
                dsu.union(i, owner[e])
            else:
                owner[e] = i
    groups: dict[int, list[str]] = defaultdict(list)
    for e, i in owner.items():
        groups[dsu.find(i)].append(e)
    return [[accounts[root][0]] + sorted(es) for root, es in groups.items()]


accounts = [
    ["John", "johnsmith@mail.com", "john_newyork@mail.com"],
    ["John", "johnsmith@mail.com", "john00@mail.com"],
    ["Mary", "mary@mail.com"],
    ["John", "johnnybravo@mail.com"],
]
out = sorted(accounts_merge(accounts))
assert out == [
    ["John", "john00@mail.com", "john_newyork@mail.com", "johnsmith@mail.com"],
    ["John", "johnnybravo@mail.com"],
    ["Mary", "mary@mail.com"],
]
print(out)

### 5.1.3 [990. Satisfiability of Equality Equations](https://leetcode.com/problems/satisfiability-of-equality-equations/) — ★ · Medium

**Problem.** Equations `a==b` / `a!=b` over letters; can all hold simultaneously?  
**Signal.** Equalities define groups (union), inequalities must cross groups.  
**Idea.** First union all `==` pairs, then check that no `!=` pair has the same root.  
**Cost.** O(n α(26))

In [ ]:
def equations_possible(equations: list[str]) -> bool:
    """True if all equations can be satisfied."""
    dsu = DSU(26)
    for eq in equations:
        if eq[1] == "=":
            dsu.union(ord(eq[0]) - 97, ord(eq[3]) - 97)
    return all(
        dsu.find(ord(eq[0]) - 97) != dsu.find(ord(eq[3]) - 97)
        for eq in equations
        if eq[1] == "!"
    )


assert not equations_possible(["a==b", "b!=a"])
assert equations_possible(["b==a", "a==b"])
assert not equations_possible(["a==b", "b==c", "a!=c"])
print("ok")

### 5.1.4 [1319. Number of Operations to Make Network Connected](https://leetcode.com/problems/number-of-operations-to-make-network-connected/) — ★ · Medium

**Problem.** Move cables (edges) to connect all `n` computers; fewest moves, or −1.  
**Signal.** Count components and spare (redundant) cables.  
**Idea.** Need at least `n − 1` cables. With enough cables, the answer is `components − 1`; each redundant edge found by union–find can be moved to join two components.  
**Cost.** O(E α(n))

In [ ]:
def make_connected(n: int, connections: list[list[int]]) -> int:
    """Cable moves needed to connect all computers."""
    if len(connections) < n - 1:
        return -1
    dsu = DSU(n)
    components = n
    for a, b in connections:
        components -= dsu.union(a, b)
    return components - 1


assert make_connected(4, [[0, 1], [0, 2], [1, 2]]) == 1
assert make_connected(6, [[0, 1], [0, 2], [0, 3], [1, 2], [1, 3]]) == 2
assert make_connected(6, [[0, 1], [0, 2], [0, 3], [1, 2]]) == -1
print("ok")

---
# 6) Weighted Shortest Paths and MST

**Dijkstra**: pop the closest unfinished node from a min-heap; its distance is final
(non-negative weights). Skip stale heap entries whose distance is outdated.

```python
dist = {src: 0}; heap = [(0, src)]
while heap:
    d, u = heapq.heappop(heap)
    if d > dist[u]: continue           # stale entry
    for v, w in graph[u]:
        if d + w < dist.get(v, inf):
            dist[v] = d + w; heapq.heappush(heap, (d + w, v))
```

The same loop with a different "cost combination" (max instead of sum, product instead
of sum) solves bottleneck and probability variants.

## 6.1 Dijkstra and variants

### 6.1.1 [743. Network Delay Time](https://leetcode.com/problems/network-delay-time/) — ★★ · Medium

**Problem.** Directed weighted edges; time for a signal from `k` to reach all nodes (−1 if not all).  
**Signal.** Single-source shortest paths, non-negative weights → Dijkstra.  
**Idea.** Run Dijkstra from `k`; the answer is the largest finalised distance, or −1 if some node is unreachable.  
**Cost.** O(E log V)

In [ ]:
def network_delay_time(times: list[list[int]], n: int, k: int) -> int:
    """Time for all nodes to receive the signal."""
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = {k: 0}
    heap = [(0, k)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, w in graph[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v] = d + w
                heapq.heappush(heap, (d + w, v))
    return max(dist.values()) if len(dist) == n else -1


assert network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
assert network_delay_time([[1, 2, 1]], 2, 2) == -1
print("ok")

### 6.1.2 [1631. Path With Minimum Effort](https://leetcode.com/problems/path-with-minimum-effort/) — ★ · Medium

**Problem.** Grid of heights; a path's effort is its largest absolute step difference. Minimise effort from top-left to bottom-right.  
**Signal.** Bottleneck shortest path → Dijkstra where path cost = max edge, not sum.  
**Idea.** Same Dijkstra loop with `new = max(d, |h1 − h2|)`. The first time the target is popped, its effort is optimal.  
**Cost.** O(mn log(mn))

In [ ]:
def minimum_effort_path(heights: list[list[int]]) -> int:
    """Least possible maximum step on a path."""
    rows, cols = len(heights), len(heights[0])
    best = {(0, 0): 0}
    heap = [(0, 0, 0)]
    while heap:
        d, r, c = heapq.heappop(heap)
        if (r, c) == (rows - 1, cols - 1):
            return d
        if d > best[(r, c)]:
            continue
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                nd = max(d, abs(heights[nr][nc] - heights[r][c]))
                if nd < best.get((nr, nc), float("inf")):
                    best[(nr, nc)] = nd
                    heapq.heappush(heap, (nd, nr, nc))
    return 0


assert minimum_effort_path([[1, 2, 2], [3, 8, 2], [5, 3, 5]]) == 2
assert minimum_effort_path([[1, 2, 3], [3, 8, 4], [5, 3, 5]]) == 1
print("ok")

### 6.1.3 [778. Swim in Rising Water](https://leetcode.com/problems/swim-in-rising-water/) — ★★ · Hard

**Problem.** At time `t` you can swim through cells with elevation ≤ `t`. Earliest time to reach the bottom-right corner?  
**Signal.** Minimise the maximum cell on the path → bottleneck Dijkstra.  
**Idea.** Pop the lowest-max frontier cell; push neighbours with `max(t, grid[nr][nc])`. When the target is popped, `t` is the answer.  
**Cost.** O(n² log n)

In [ ]:
def swim_in_water(grid: list[list[int]]) -> int:
    """Earliest time to reach the far corner."""
    n = len(grid)
    heap = [(grid[0][0], 0, 0)]
    seen = {(0, 0)}
    while heap:
        t, r, c = heapq.heappop(heap)
        if (r, c) == (n - 1, n - 1):
            return t
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and (nr, nc) not in seen:
                seen.add((nr, nc))
                heapq.heappush(heap, (max(t, grid[nr][nc]), nr, nc))
    return -1


assert swim_in_water([[0, 2], [1, 3]]) == 3
g = [
    [0, 1, 2, 3, 4],
    [24, 23, 22, 21, 5],
    [12, 13, 14, 15, 16],
    [11, 17, 18, 19, 20],
    [10, 9, 8, 7, 6],
]
assert swim_in_water(g) == 16
print("ok")

### 6.1.4 [1514. Path with Maximum Probability](https://leetcode.com/problems/path-with-maximum-probability/) — ★ · Medium

**Problem.** Undirected edges with success probabilities; maximise the probability of reaching `end`.  
**Signal.** Dijkstra with products and a max-heap.  
**Idea.** Probabilities multiply and never increase along a path, so the greedy "best first" argument of Dijkstra holds. Push negated probabilities onto the min-heap.  
**Cost.** O(E log V)

In [ ]:
def max_probability(
    n: int, edges: list[list[int]], succ_prob: list[float], start: int, end: int
) -> float:
    """Highest success probability from start to end."""
    graph = defaultdict(list)
    for (a, b), p in zip(edges, succ_prob):
        graph[a].append((b, p))
        graph[b].append((a, p))
    best = [0.0] * n
    best[start] = 1.0
    heap = [(-1.0, start)]
    while heap:
        p, u = heapq.heappop(heap)
        p = -p
        if u == end:
            return p
        if p < best[u]:
            continue
        for v, w in graph[u]:
            if p * w > best[v]:
                best[v] = p * w
                heapq.heappush(heap, (-best[v], v))
    return 0.0


assert max_probability(3, [[0, 1], [1, 2], [0, 2]], [0.5, 0.5, 0.2], 0, 2) == 0.25
assert max_probability(3, [[0, 1]], [0.5], 0, 2) == 0.0
print("ok")

### 6.1.5 [787. Cheapest Flights Within K Stops](https://leetcode.com/problems/cheapest-flights-within-k-stops/) — ★★ · Medium

**Problem.** Cheapest price from `src` to `dst` using at most `k` stops (k + 1 flights).  
**Signal.** Edge-count limit → Bellman–Ford for `k + 1` rounds.  
**Idea.** Round `i` relaxes every edge using the *previous* round's prices (copy the array), so after round `i` prices use at most `i` flights. Dijkstra on cost alone fails because a cheaper path may use too many stops.  
**Cost.** O(k · E)

In [ ]:
def find_cheapest_price(
    n: int, flights: list[list[int]], src: int, dst: int, k: int
) -> int:
    """Cheapest cost with at most k stops, or -1."""
    inf = float("inf")
    prices = [inf] * n
    prices[src] = 0
    for _ in range(k + 1):
        nxt = prices.copy()
        for u, v, w in flights:
            if prices[u] + w < nxt[v]:
                nxt[v] = prices[u] + w
        prices = nxt
    return -1 if prices[dst] == inf else prices[dst]


flights = [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]]
assert find_cheapest_price(4, flights, 0, 3, 1) == 700
assert find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1) == 200
assert find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0) == 500
print("ok")

### 6.1.6 [1584. Min Cost to Connect All Points](https://leetcode.com/problems/min-cost-to-connect-all-points/) — ★★ · Medium

**Problem.** Connect all points with minimum total Manhattan distance.  
**Signal.** Minimum spanning tree on a complete graph → Prim's algorithm.  
**Idea.** Grow the tree from point 0; keep `best[i]` = cheapest edge from the tree to `i`. Each step add the closest outside point and update `best` with distances from it. The O(n²) array version beats a heap on a dense graph.  
**Cost.** O(n²) time · O(n) space

In [ ]:
def min_cost_connect_points(points: list[list[int]]) -> int:
    """MST weight under Manhattan distance."""
    n = len(points)
    best = [float("inf")] * n
    in_tree = [False] * n
    best[0] = 0
    total = 0
    for _ in range(n):
        u = min((i for i in range(n) if not in_tree[i]), key=best.__getitem__)
        in_tree[u] = True
        total += best[u]
        for v in range(n):
            if not in_tree[v]:
                d = abs(points[u][0] - points[v][0]) + abs(points[u][1] - points[v][1])
                best[v] = min(best[v], d)
    return total


assert min_cost_connect_points([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]) == 20
assert min_cost_connect_points([[3, 12], [-2, 5], [-4, 1]]) == 18
print("ok")

### 6.1.7 [332. Reconstruct Itinerary](https://leetcode.com/problems/reconstruct-itinerary/) — ★★ · Hard

**Problem.** Use every ticket once starting from `"JFK"`; return the lexicographically smallest itinerary.  
**Signal.** Use every edge exactly once → Eulerian path (Hierholzer's algorithm).  
**Idea.** Sort each airport's destinations. DFS: repeatedly take the smallest unused ticket; when an airport has no tickets left, append it to the route. The route is built in reverse, which correctly handles dead ends taken too early.  
**Cost.** O(E log E)

In [ ]:
def find_itinerary(tickets: list[list[str]]) -> list[str]:
    """Smallest-lexical itinerary using every ticket."""
    graph = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):
        graph[a].append(b)
    route: list[str] = []
    stack = ["JFK"]
    while stack:
        while graph[stack[-1]]:
            stack.append(graph[stack[-1]].pop())
        route.append(stack.pop())
    return route[::-1]


assert find_itinerary(
    [["MUC", "LHR"], ["JFK", "MUC"], ["SFO", "SJC"], ["LHR", "SFO"]]
) == [
    "JFK",
    "MUC",
    "LHR",
    "SFO",
    "SJC",
]
tickets = [
    ["JFK", "SFO"],
    ["JFK", "ATL"],
    ["SFO", "ATL"],
    ["ATL", "JFK"],
    ["ATL", "SFO"],
]
assert find_itinerary(tickets) == ["JFK", "ATL", "JFK", "SFO", "ATL", "SFO"]
assert find_itinerary([["JFK", "KUL"], ["JFK", "NRT"], ["NRT", "JFK"]]) == [
    "JFK",
    "NRT",
    "JFK",
    "KUL",
]
print("ok")

---
# 7) Backtracking

```python
def backtrack(start, path):
    if is_complete(path):
        out.append(path.copy()); return
    for i in range(start, len(choices)):
        if not valid(choices[i]): continue          # prune
        path.append(choices[i])                     # choose
        backtrack(i + 1, path)                      # explore (i for reuse, 0 for permutations)
        path.pop()                                  # un-choose
```

| Problem type | Recurse with | Avoid duplicates by |
|---|---|---|
| subsets / combinations | `i + 1` | sorting + skip `nums[i] == nums[i-1]` when `i > start` |
| unlimited reuse | `i` | — |
| permutations | 0 with a `used` set | sorting + skip if previous equal value is unused |

## 7.1 Subsets, combinations and permutations

### 7.1.1 [78. Subsets](https://leetcode.com/problems/subsets/) — ★★ · Medium

**Problem.** All subsets of distinct integers.  
**Signal.** Each element is in or out → backtracking (or iterate: double the list per element).  
**Idea.** At each recursion level, record the current path (every node of the decision tree is a subset), then try extending it with each later element.  
**Cost.** O(n · 2ⁿ)

In [ ]:
def subsets(nums: list[int]) -> list[list[int]]:
    """Power set."""
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int) -> None:
        out.append(path.copy())
        for i in range(start, len(nums)):
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return out


assert subsets([1, 2, 3]) == [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
print(len(subsets([1, 2, 3, 4])))

### 7.1.2 [90. Subsets II](https://leetcode.com/problems/subsets-ii/) — ★★ · Medium

**Problem.** All unique subsets when the input has duplicates.  
**Signal.** Sort, then skip equal siblings at the same depth.  
**Idea.** After sorting, at one recursion level choosing the second of two equal values would create the same subset as choosing the first, so skip `nums[i] == nums[i−1]` when `i > start`.  
**Cost.** O(n · 2ⁿ)

In [ ]:
def subsets_with_dup(nums: list[int]) -> list[list[int]]:
    """Unique subsets of a multiset."""
    nums = sorted(nums)
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int) -> None:
        out.append(path.copy())
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return out


assert subsets_with_dup([1, 2, 2]) == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
print(subsets_with_dup([1, 2, 2]))

### 7.1.3 [77. Combinations](https://leetcode.com/problems/combinations/) — ★★ · Medium

**Problem.** All combinations of `k` numbers from `1..n`.  
**Signal.** Subsets restricted to size `k`, with pruning.  
**Idea.** Record when the path has `k` elements. Prune: stop the loop when too few numbers remain to reach size `k`.  
**Cost.** O(k · C(n, k))

In [ ]:
def combine(n: int, k: int) -> list[list[int]]:
    """All k-combinations of 1..n."""
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int) -> None:
        if len(path) == k:
            out.append(path.copy())
            return
        for i in range(start, n - (k - len(path)) + 2):
            path.append(i)
            backtrack(i + 1)
            path.pop()

    backtrack(1)
    return out


assert combine(4, 2) == [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
assert combine(1, 1) == [[1]]
print(combine(4, 2))

### 7.1.4 [39. Combination Sum](https://leetcode.com/problems/combination-sum/) — ★★ · Medium

**Problem.** Distinct candidates, unlimited reuse; all combinations summing to `target`.  
**Signal.** Reuse allowed → recurse with `i` (not `i + 1`); sorted input lets you prune.  
**Idea.** Sort. Subtract the candidate from the remaining target; if a candidate exceeds the remainder, every later one does too, so break.  
**Cost.** exponential, bounded by target / min

In [ ]:
def combination_sum(candidates: list[int], target: int) -> list[list[int]]:
    """Combinations with reuse summing to target."""
    candidates = sorted(candidates)
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int, remaining: int) -> None:
        if remaining == 0:
            out.append(path.copy())
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining:
                break
            path.append(candidates[i])
            backtrack(i, remaining - candidates[i])
            path.pop()

    backtrack(0, target)
    return out


assert combination_sum([2, 3, 6, 7], 7) == [[2, 2, 3], [7]]
assert combination_sum([2, 3, 5], 8) == [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
assert combination_sum([2], 1) == []
print(combination_sum([2, 3, 6, 7], 7))

### 7.1.5 [40. Combination Sum II](https://leetcode.com/problems/combination-sum-ii/) — ★★ · Medium

**Problem.** Candidates may repeat; each used at most once; unique combinations summing to `target`.  
**Signal.** No reuse (`i + 1`) + duplicate skipping (Subsets II rule).  
**Idea.** Sort; skip `candidates[i] == candidates[i−1]` when `i > start`; break when a candidate exceeds the remainder.  
**Cost.** O(2ⁿ) worst

In [ ]:
def combination_sum2(candidates: list[int], target: int) -> list[list[int]]:
    """Unique single-use combinations summing to target."""
    candidates = sorted(candidates)
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int, remaining: int) -> None:
        if remaining == 0:
            out.append(path.copy())
            return
        for i in range(start, len(candidates)):
            if i > start and candidates[i] == candidates[i - 1]:
                continue
            if candidates[i] > remaining:
                break
            path.append(candidates[i])
            backtrack(i + 1, remaining - candidates[i])
            path.pop()

    backtrack(0, target)
    return out


assert combination_sum2([10, 1, 2, 7, 6, 1, 5], 8) == [
    [1, 1, 6],
    [1, 2, 5],
    [1, 7],
    [2, 6],
]
assert combination_sum2([2, 5, 2, 1, 2], 5) == [[1, 2, 2], [5]]
print(combination_sum2([10, 1, 2, 7, 6, 1, 5], 8))

### 7.1.6 [216. Combination Sum III](https://leetcode.com/problems/combination-sum-iii/) — ★ · Medium

**Problem.** All combinations of `k` distinct digits 1–9 summing to `n`.  
**Signal.** Combinations with both a size and a sum constraint.  
**Idea.** Backtrack over 1–9 with `i + 1`; stop when the path has `k` digits, record if the sum hits zero; break when a digit exceeds the remainder.  
**Cost.** O(C(9, k))

In [ ]:
def combination_sum3(k: int, n: int) -> list[list[int]]:
    """k distinct digits summing to n."""
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int, remaining: int) -> None:
        if len(path) == k:
            if remaining == 0:
                out.append(path.copy())
            return
        for d in range(start, 10):
            if d > remaining:
                break
            path.append(d)
            backtrack(d + 1, remaining - d)
            path.pop()

    backtrack(1, n)
    return out


assert combination_sum3(3, 7) == [[1, 2, 4]]
assert combination_sum3(3, 9) == [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
assert combination_sum3(4, 1) == []
print(combination_sum3(3, 9))

### 7.1.7 [46. Permutations](https://leetcode.com/problems/permutations/) — ★★ · Medium

**Problem.** All permutations of distinct integers.  
**Signal.** Order matters → loop from 0 each level, skip used elements.  
**Idea.** Keep a `used` flag per index; the path is complete at length `n`. (Swap-based in-place permutation is an alternative.)  
**Cost.** O(n · n!)

In [ ]:
def permute(nums: list[int]) -> list[list[int]]:
    """All orderings of nums."""
    out: list[list[int]] = []
    path: list[int] = []
    used = [False] * len(nums)

    def backtrack() -> None:
        if len(path) == len(nums):
            out.append(path.copy())
            return
        for i, x in enumerate(nums):
            if not used[i]:
                used[i] = True
                path.append(x)
                backtrack()
                path.pop()
                used[i] = False

    backtrack()
    return out


assert permute([1, 2, 3]) == [
    [1, 2, 3],
    [1, 3, 2],
    [2, 1, 3],
    [2, 3, 1],
    [3, 1, 2],
    [3, 2, 1],
]
assert permute([1]) == [[1]]
print(len(permute([1, 2, 3, 4])))

### 7.1.8 [47. Permutations II](https://leetcode.com/problems/permutations-ii/) — ★ · Medium

**Problem.** All unique permutations when the input has duplicates.  
**Signal.** Sort + skip a value equal to the previous one if the previous copy is unused.  
**Idea.** Among equal values, force them to be used left to right: skip index `i` when `nums[i] == nums[i−1]` and `nums[i−1]` is not currently used.  
**Cost.** O(n · n!)

In [ ]:
def permute_unique(nums: list[int]) -> list[list[int]]:
    """Unique orderings of a multiset."""
    nums = sorted(nums)
    out: list[list[int]] = []
    path: list[int] = []
    used = [False] * len(nums)

    def backtrack() -> None:
        if len(path) == len(nums):
            out.append(path.copy())
            return
        for i, x in enumerate(nums):
            if used[i] or (i and nums[i] == nums[i - 1] and not used[i - 1]):
                continue
            used[i] = True
            path.append(x)
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return out


assert permute_unique([1, 1, 2]) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
assert len(permute_unique([1, 2, 3])) == 6
print(permute_unique([1, 1, 2]))

### 7.1.9 [17. Letter Combinations of a Phone Number](https://leetcode.com/problems/letter-combinations-of-a-phone-number/) — ★★ · Medium

**Problem.** All letter strings a phone digit string could represent.  
**Signal.** One choice per position from a fixed mapping → backtracking (Cartesian product).  
**Idea.** Recurse over positions; at each digit, try each mapped letter.  
**Cost.** O(4ⁿ · n)

In [ ]:
PHONE = {
    "2": "abc",
    "3": "def",
    "4": "ghi",
    "5": "jkl",
    "6": "mno",
    "7": "pqrs",
    "8": "tuv",
    "9": "wxyz",
}


def letter_combinations(digits: str) -> list[str]:
    """Letter strings for a digit string."""
    if not digits:
        return []
    out: list[str] = []

    def backtrack(i: int, path: str) -> None:
        if i == len(digits):
            out.append(path)
            return
        for ch in PHONE[digits[i]]:
            backtrack(i + 1, path + ch)

    backtrack(0, "")
    return out


assert letter_combinations("23") == [
    "ad",
    "ae",
    "af",
    "bd",
    "be",
    "bf",
    "cd",
    "ce",
    "cf",
]
assert letter_combinations("") == []
print(letter_combinations("23"))

### 7.1.10 [22. Generate Parentheses](https://leetcode.com/problems/generate-parentheses/) — ★★ · Medium

**Problem.** All well-formed strings of `n` pairs of parentheses.  
**Signal.** Choices constrained by counts → backtrack with `open` and `close` counters.  
**Idea.** Add `(` while `open < n`; add `)` while `close < open`. Every complete string of length `2n` produced this way is valid, and no invalid prefix is ever explored.  
**Cost.** O(4ⁿ / √n) (Catalan number)

In [ ]:
def generate_parenthesis(n: int) -> list[str]:
    """All balanced strings of n pairs."""
    out: list[str] = []

    def backtrack(path: str, open_count: int, close_count: int) -> None:
        if len(path) == 2 * n:
            out.append(path)
            return
        if open_count < n:
            backtrack(path + "(", open_count + 1, close_count)
        if close_count < open_count:
            backtrack(path + ")", open_count, close_count + 1)

    backtrack("", 0, 0)
    return out


assert generate_parenthesis(3) == ["((()))", "(()())", "(())()", "()(())", "()()()"]
assert generate_parenthesis(1) == ["()"]
print(generate_parenthesis(3))

### 7.1.11 [1239. Maximum Length of a Concatenated String with Unique Characters](https://leetcode.com/problems/maximum-length-of-a-concatenated-string-with-unique-characters/) — ★ · Medium

**Problem.** Choose a subsequence of strings whose concatenation has all-unique characters; maximise its length.  
**Signal.** Subset enumeration with a feasibility check → backtracking with bitmasks.  
**Idea.** Discard strings with internal duplicates; represent each as a 26-bit mask. Backtrack: include a string only if its mask does not overlap the current mask.  
**Cost.** O(2ⁿ)

In [ ]:
def max_length(arr: list[str]) -> int:
    """Longest unique-char concatenation length."""
    masks = []
    for s in arr:
        m = 0
        for ch in s:
            m |= 1 << (ord(ch) - 97)
        if bin(m).count("1") == len(s):
            masks.append(m)

    def backtrack(i: int, used: int) -> int:
        best = bin(used).count("1")
        for j in range(i, len(masks)):
            if not used & masks[j]:
                best = max(best, backtrack(j + 1, used | masks[j]))
        return best

    return backtrack(0, 0)


assert max_length(["un", "iq", "ue"]) == 4
assert max_length(["cha", "r", "act", "ers"]) == 6
assert max_length(["abcdefghijklmnopqrstuvwxyz"]) == 26
print(max_length(["cha", "r", "act", "ers"]))

## 7.2 Search with constraints

### 7.2.1 [79. Word Search](https://leetcode.com/problems/word-search/) — ★★★ · Medium

**Problem.** Does the word exist in the grid as a path of adjacent cells (no reuse)?  
**Signal.** Path search with "no revisit" → DFS with temporary marking and backtracking.  
**Idea.** From each cell matching the first letter, DFS: mark the cell, try 4 neighbours for the next letter, unmark on return. Prune quickly: count letters first — if the grid lacks enough of some letter, return early.  
**Cost.** O(cells · 3^L)

In [ ]:
def exist(board: list[list[str]], word: str) -> bool:
    """True if word can be traced in the grid."""
    rows, cols = len(board), len(board[0])

    def dfs(r: int, c: int, i: int) -> bool:
        if i == len(word):
            return True
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False
        board[r][c] = "#"
        found = any(dfs(r + dr, c + dc, i + 1) for dr, dc in DIRS)
        board[r][c] = word[i]
        return found

    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))


b = [list("ABCE"), list("SFCS"), list("ADEE")]
assert exist(b, "ABCCED") and exist(b, "SEE") and not exist(b, "ABCB")
print("ok")

### 7.2.2 [131. Palindrome Partitioning](https://leetcode.com/problems/palindrome-partitioning/) — ★★ · Medium

**Problem.** All ways to split `s` into palindromic pieces.  
**Signal.** Choose the next cut; only continue if the piece is a palindrome.  
**Idea.** At position `start`, try each end `i`; if `s[start:i+1]` is a palindrome, add it and recurse from `i + 1`. (Precomputing a palindrome table removes repeated checks.)  
**Cost.** O(n · 2ⁿ)

In [ ]:
def partition_palindromes(s: str) -> list[list[str]]:
    """Every palindromic partition of s."""
    out: list[list[str]] = []
    path: list[str] = []

    def backtrack(start: int) -> None:
        if start == len(s):
            out.append(path.copy())
            return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if piece == piece[::-1]:
                path.append(piece)
                backtrack(end)
                path.pop()

    backtrack(0)
    return out


assert partition_palindromes("aab") == [["a", "a", "b"], ["aa", "b"]]
assert partition_palindromes("a") == [["a"]]
print(partition_palindromes("aab"))

### 7.2.3 [93. Restore IP Addresses](https://leetcode.com/problems/restore-ip-addresses/) — ★ · Medium

**Problem.** All valid IPv4 addresses formed by inserting three dots into a digit string.  
**Signal.** Four segments, each 1–3 digits, value ≤ 255, no leading zeros.  
**Idea.** Backtrack segment by segment; prune when the remaining digits cannot fill the remaining segments (each needs 1–3 digits).  
**Cost.** O(3⁴)

In [ ]:
def restore_ip_addresses(s: str) -> list[str]:
    """All valid IPs from the digit string."""
    out: list[str] = []

    def backtrack(start: int, parts: list[str]) -> None:
        if len(parts) == 4:
            if start == len(s):
                out.append(".".join(parts))
            return
        remaining = len(s) - start
        if not 4 - len(parts) <= remaining <= 3 * (4 - len(parts)):
            return
        for length in range(1, 4):
            seg = s[start : start + length]
            if len(seg) < length or (seg[0] == "0" and length > 1) or int(seg) > 255:
                break
            backtrack(start + length, parts + [seg])

    backtrack(0, [])
    return out


assert restore_ip_addresses("25525511135") == ["255.255.11.135", "255.255.111.35"]
assert restore_ip_addresses("0000") == ["0.0.0.0"]
assert restore_ip_addresses("101023") == [
    "1.0.10.23",
    "1.0.102.3",
    "10.1.0.23",
    "10.10.2.3",
    "101.0.2.3",
]
print(restore_ip_addresses("25525511135"))

### 7.2.4 [51. N-Queens](https://leetcode.com/problems/n-queens/) — ★★ · Hard

**Problem.** All placements of `n` non-attacking queens on an `n × n` board.  
**Signal.** One queen per row; track used columns and both diagonals in sets.  
**Idea.** Place row by row. A square `(r, c)` is attacked if its column, `r − c` diagonal or `r + c` anti-diagonal is already used. Add to the sets, recurse, remove.  
**Cost.** O(n!)

In [ ]:
def solve_n_queens(n: int) -> list[list[str]]:
    """All N-Queens boards."""
    out: list[list[str]] = []
    cols: set[int] = set()
    diag: set[int] = set()
    anti: set[int] = set()
    queens: list[int] = []

    def backtrack(r: int) -> None:
        if r == n:
            out.append(["." * c + "Q" + "." * (n - c - 1) for c in queens])
            return
        for c in range(n):
            if c in cols or r - c in diag or r + c in anti:
                continue
            cols.add(c)
            diag.add(r - c)
            anti.add(r + c)
            queens.append(c)
            backtrack(r + 1)
            queens.pop()
            cols.remove(c)
            diag.remove(r - c)
            anti.remove(r + c)

    backtrack(0)
    return out


assert solve_n_queens(4) == [
    [".Q..", "...Q", "Q...", "..Q."],
    ["..Q.", "Q...", "...Q", ".Q.."],
]
assert len(solve_n_queens(1)) == 1
print(len(solve_n_queens(8)))

### 7.2.5 [52. N-Queens II](https://leetcode.com/problems/n-queens-ii/) — ★★ · Hard

**Problem.** Count the N-Queens solutions.  
**Signal.** Same search, count instead of building boards; bitmasks make it fast.  
**Idea.** Represent attacked columns and diagonals as bitmasks. At each row, free squares are `~(cols | diag | anti) & full`; take the lowest set bit repeatedly. Shift the diagonal masks by one for the next row.  
**Cost.** O(n!) with tiny constants

In [ ]:
def total_n_queens(n: int) -> int:
    """Number of N-Queens solutions."""
    full = (1 << n) - 1

    def count(cols: int, diag: int, anti: int) -> int:
        if cols == full:
            return 1
        total = 0
        free = ~(cols | diag | anti) & full
        while free:
            bit = free & -free
            free -= bit
            total += count(cols | bit, ((diag | bit) << 1) & full, (anti | bit) >> 1)
        return total

    return count(0, 0, 0)


assert [total_n_queens(n) for n in range(1, 9)] == [1, 0, 0, 2, 10, 4, 40, 92]
print(total_n_queens(8))

### 7.2.6 [37. Sudoku Solver](https://leetcode.com/problems/sudoku-solver/) — ★★ · Hard

**Problem.** Fill a 9×9 Sudoku in place.  
**Signal.** Constraint search: try digits in empty cells with row/column/box sets.  
**Idea.** Collect empty cells and per-row/column/box used digits. Recursively fill the next empty cell with each allowed digit; undo on failure. Choosing the cell with the fewest options first (MRV) speeds it up dramatically.  
**Cost.** exponential worst case, fast in practice

In [ ]:
def solve_sudoku(board: list[list[str]]) -> None:
    """Solve the puzzle in place."""
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empty = []
    for r in range(9):
        for c in range(9):
            d = board[r][c]
            if d == ".":
                empty.append((r, c))
            else:
                rows[r].add(d)
                cols[c].add(d)
                boxes[r // 3 * 3 + c // 3].add(d)

    def options(r: int, c: int) -> set[str]:
        return set("123456789") - rows[r] - cols[c] - boxes[r // 3 * 3 + c // 3]

    def backtrack() -> bool:
        if not empty:
            return True
        best = min(range(len(empty)), key=lambda i: len(options(*empty[i])))
        empty[best], empty[-1] = empty[-1], empty[best]
        r, c = empty.pop()
        for d in options(r, c):
            board[r][c] = d
            rows[r].add(d)
            cols[c].add(d)
            boxes[r // 3 * 3 + c // 3].add(d)
            if backtrack():
                return True
            rows[r].remove(d)
            cols[c].remove(d)
            boxes[r // 3 * 3 + c // 3].remove(d)
        board[r][c] = "."
        empty.append((r, c))
        return False

    backtrack()


puzzle = [
    "53..7....",
    "6..195...",
    ".98....6.",
    "8...6...3",
    "4..8.3..1",
    "7...2...6",
    ".6....28.",
    "...419..5",
    "....8..79",
]
board = [list(row) for row in puzzle]
solve_sudoku(board)
assert "".join(board[0]) == "534678912" and "".join(board[8]) == "345286179"
print("\n".join("".join(r) for r in board))

### 7.2.7 [698. Partition to K Equal Sum Subsets](https://leetcode.com/problems/partition-to-k-equal-sum-subsets/) — ★ · Medium

**Problem.** Can the array be split into `k` subsets with equal sums?  
**Signal.** Fill k buckets with backtracking; sort descending and skip symmetric states.  
**Idea.** Target per bucket is `sum / k`. Place large numbers first (fail fast). For each number, try every bucket that still has room; if a bucket is empty and placing fails, don't try other empty buckets (they are equivalent).  
**Cost.** O(kⁿ) worst, heavily pruned

In [ ]:
def can_partition_k_subsets(nums: list[int], k: int) -> bool:
    """True if nums splits into k equal-sum groups."""
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    nums = sorted(nums, reverse=True)
    if nums[0] > target:
        return False
    buckets = [0] * k

    def backtrack(i: int) -> bool:
        if i == len(nums):
            return True
        for b in range(k):
            if buckets[b] + nums[i] <= target:
                buckets[b] += nums[i]
                if backtrack(i + 1):
                    return True
                buckets[b] -= nums[i]
            if buckets[b] == 0:
                break
        return False

    return backtrack(0)


assert can_partition_k_subsets([4, 3, 2, 3, 5, 2, 1], 4)
assert not can_partition_k_subsets([1, 2, 3, 4], 3)
print("ok")

### 7.2.8 [473. Matchsticks to Square](https://leetcode.com/problems/matchsticks-to-square/) — ★ · Medium

**Problem.** Use all matchsticks to form a square.  
**Signal.** Partition into 4 equal-sum subsets.  
**Idea.** Exactly the k-bucket backtracking with `k = 4`.  
**Cost.** O(4ⁿ) worst, pruned

In [ ]:
def makesquare(matchsticks: list[int]) -> bool:
    """True if the sticks form a square."""
    return len(matchsticks) >= 4 and can_partition_k_subsets(matchsticks, 4)


assert makesquare([1, 1, 2, 2, 2])
assert not makesquare([3, 3, 3, 3, 4])
print("ok")

### 7.2.9 [282. Expression Add Operators](https://leetcode.com/problems/expression-add-operators/) — ★ · Hard

**Problem.** Insert `+`, `−`, `*` between digits of `num` so the expression equals `target`.  
**Signal.** Backtracking that tracks the last multiplicative term to undo it for `*`.  
**Idea.** Carry `value` (total so far) and `last` (last term). For `+x`: `value + x`, last `x`. For `−x`: `value − x`, last `−x`. For `*x`: `value − last + last*x`, last `last*x` — this re-applies precedence. Skip operands with leading zeros.  
**Cost.** O(4ⁿ)

In [ ]:
def add_operators(num: str, target: int) -> list[str]:
    """Expressions over num that evaluate to target."""
    out: list[str] = []

    def backtrack(i: int, expr: str, value: int, last: int) -> None:
        if i == len(num):
            if value == target:
                out.append(expr)
            return
        for j in range(i + 1, len(num) + 1):
            part = num[i:j]
            if len(part) > 1 and part[0] == "0":
                break
            x = int(part)
            if i == 0:
                backtrack(j, part, x, x)
            else:
                backtrack(j, f"{expr}+{part}", value + x, x)
                backtrack(j, f"{expr}-{part}", value - x, -x)
                backtrack(j, f"{expr}*{part}", value - last + last * x, last * x)

    backtrack(0, "", 0, 0)
    return out


assert sorted(add_operators("123", 6)) == ["1*2*3", "1+2+3"]
assert sorted(add_operators("232", 8)) == ["2*3+2", "2+3*2"]
assert add_operators("3456237490", 9191) == []
assert sorted(add_operators("105", 5)) == ["1*0+5", "10-5"]
print(add_operators("123", 6))

### 7.2.10 [301. Remove Invalid Parentheses](https://leetcode.com/problems/remove-invalid-parentheses/) — ★ · Hard

**Problem.** Remove the minimum number of parentheses to make the string valid; return all results.  
**Signal.** Minimum removals → BFS by number of removals; first level with a valid string wins.  
**Idea.** Level 0 is the string itself. Generate the next level by deleting one parenthesis in every possible position (as a set to dedupe). Stop at the first level containing valid strings and return those.  
**Cost.** exponential worst case, fine for n ≤ 25

In [ ]:
def remove_invalid_parentheses(s: str) -> list[str]:
    """All valid strings after minimum removals."""

    def valid(t: str) -> bool:
        balance = 0
        for ch in t:
            balance += ch == "("
            balance -= ch == ")"
            if balance < 0:
                return False
        return balance == 0

    level = {s}
    while True:
        good = [t for t in level if valid(t)]
        if good:
            return sorted(good)
        level = {
            t[:i] + t[i + 1 :] for t in level for i in range(len(t)) if t[i] in "()"
        }


assert remove_invalid_parentheses("()())()") == ["(())()", "()()()"]
assert remove_invalid_parentheses("(a)())()") == ["(a())()", "(a)()()"]
assert remove_invalid_parentheses(")(") == [""]
print(remove_invalid_parentheses("()())()"))

---
# 8) Conclusion

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Grid DFS | islands, regions, flood fill | mark visited before recursing | O(mn) |
| Reverse search | border-connected, ocean flow | start from the target side and invert | O(mn) |
| Multi-source BFS | distance to nearest X, spreading | enqueue all sources at distance 0 | O(mn) |
| BFS on states | word ladder, lock, buses | generate neighbours on the fly | O(states) |
| Topological sort | prerequisites, ordering, alien alphabet | Kahn: peel in-degree 0 | O(V + E) |
| Leaf peeling | tree centres | repeatedly remove degree-1 nodes | O(n) |
| Union–find | redundant edge, merging groups, equations | `find` with compression, union by size | ~O(1) |
| Dijkstra | non-negative weighted shortest path | min-heap, skip stale entries | O(E log V) |
| Bottleneck Dijkstra | minimise max edge on a path | `max` instead of `+` | O(E log V) |
| Bellman–Ford (k rounds) | at most k stops | relax all edges from the previous round's copy | O(kE) |
| MST (Prim) | connect all points cheaply | grow tree by cheapest outside node | O(n²) dense |
| Eulerian path | use every edge once | Hierholzer, build route in reverse | O(E log E) |
| Backtracking | all subsets / permutations / boards | choose → recurse → un-choose, prune early | exponential |